<a href="https://colab.research.google.com/github/felipeharker/alexandria-lab-jupyter/blob/main/gemini_render_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# INSTALL GOOGLE AI LIBRARIES
!pip install -q -U google-genai pillow


In [ ]:
# ACCESS AND LINK API KEY
# Prefer Colab Secrets, then environment variable, then manual fallback.
import os

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = None

if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "API KEY API KEY API KEY")

if not GEMINI_API_KEY or GEMINI_API_KEY == "API KEY API KEY API KEY":
    print("⚠️ Update GEMINI_API_KEY before running generation.")
else:
    print("✅ API key loaded.")


In [ ]:
# LOAD SYSTEM INSTRUCTIONS
SYSTEM_INSTRUCTIONS_PATH = "/content/drive/MyDrive/alex-llm-docs/system-instructions/sys-inst_img-rend_gen_v3.md"

with open(SYSTEM_INSTRUCTIONS_PATH, "r", encoding="utf-8") as f:
    system_instruction = f.read()

print(f"✅ Loaded system instructions from: {SYSTEM_INSTRUCTIONS_PATH}")


In [ ]:
# PROVIDE IMAGE FILE PATHS
PATH_RENDER  = "/content/drive/MyDrive/alex-llm-docs/sample-render-materials/KITCH RUM DAY.jpeg"  # paste path or leave blank
PATH_ELEV    = ""  # paste path or leave blank
PATH_CONTEXT = "/content/drive/MyDrive/alex-llm-docs/context-images/pylon-context-day-v2.1.jpg"  # paste path or leave blank


In [ ]:
# WRITE A TEXT PROMPT
prompt = "CREATE A PROFESSIONAL ARCHITECTURAL RENDER"


In [ ]:
# BUILD MODEL CONTENTS (IMAGE DESCRIPTIONS + FILES + PROMPT)
import os
from PIL import Image

contents = []
missing_paths = []

image_inputs = [
    ("IMAGE 1 (DESIGN RENDER): Photorealistic design render.", PATH_RENDER),
    ("IMAGE 2 (ELEVATIONS): Dimensioned 2D drawings.", PATH_ELEV),
    ("IMAGE 3 (CONTEXT): Site environment and surroundings. Preserve this context in the final render.", PATH_CONTEXT),
]

for description, path in image_inputs:
    if not path:
        continue
    if not os.path.exists(path):
        missing_paths.append(path)
        continue

    contents.append(description)
    contents.append(Image.open(path))

if not prompt.strip():
    raise ValueError("Prompt is empty. Please provide a text prompt before running generation.")

contents.append(prompt)

print(f"✅ Added {len(contents) - 1} non-text items plus the final prompt.")
if missing_paths:
    print("⚠️ Missing files:")
    for path in missing_paths:
        print(f" - {path}")


In [ ]:
# RUN THE MODEL
from google import genai
from google.genai import types
from google.colab import files
from datetime import datetime

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents=contents,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
    ),
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        filename = f"generated_image_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        image.save(filename)
        files.download(filename)
        print(f"✅ Saved image: {filename}")
